In [25]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
import os

# 🧠 LLM Session Memory (Conversation History)
This notebook demonstrates how to give an LLM "Short-Term Memory." By default, LLMs are **stateless**, meaning they forget everything the moment a response is sent. This project implements a system to track and recall previous messages in a conversation.

## 🎯 The Problem: Statelessness
If you tell an LLM *"My name is Alice"* in one request, and then ask *"What is my name?"* in the next, the LLM will not know the answer because every request is independent.

## 🛠️ The Solution: Session Memory
To solve this, we must:
1. **Store**: Save every user message and AI response in a "Store" (database or dictionary).
2. **Inject**: Every time a new question is asked, we retrieve the history for that specific user and feed it back into the LLM.
3. **Identify**: Use a `session_id` to make sure Alice's history doesn't get mixed up with Bob's history.

## ⚙️ Key Components Used
- `MessagesPlaceholder`: A "slot" in the prompt where the history is inserted.
- `InMemoryChatMessageHistory`: A simple way to store messages in RAM.
- `RunnableWithMessageHistory`: The LangChain wrapper that automates the "Retrieve $\rightarrow$ Inject $\rightarrow$ Save" loop.

In [26]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    api_key = os.environ['GeminiAPIKey'],
    temperature=0
    )

In [27]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

chain = prompt | llm | StrOutputParser()

### 1. Designing the Memory-Aware Prompt
**What**: Using `MessagesPlaceholder(variable_name="history")`.
**Why**: Unlike a standard prompt, this tells LangChain: *"Insert the entire conversation history right here before the current user input."*
**How**: The prompt structure becomes: `System Message` $\rightarrow$ `Conversation History` $\rightarrow$ `Current Human Input`.

In [28]:
store = {}
def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]




### 2. The Session Store
**What**: A Python dictionary (`store = {}`) and a helper function `get_session_history`.
**Why**: We need a way to keep different conversations separate. `session_id` acts like a "room number"—everyone in "user_1" shares a memory, but "user_2" has a clean slate.
**How**: If a `session_id` doesn't exist in the store, we create a new `InMemoryChatMessageHistory` object for it.

In [29]:
chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)




c:\Users\santh\AppData\Local\Programs\Python\Python314\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


### 3. Automating Memory with `RunnableWithMessageHistory`
**What**: Wrapping the original chain in `RunnableWithMessageHistory`.
**Why**: Without this, you would have to manually fetch history, add it to the prompt, and manually save the response every time. This wrapper does all that work automatically.
**How**: It connects the `chain` (the brain), the `get_session_history` (the memory fetcher), and defines which keys in the input correspond to the message and the history.

In [30]:
user1 = chain_with_memory.invoke(
    {"input": "My name is Alice."},
    config={"configurable": {"session_id": "user_1"}}
)


c:\Users\santh\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


### 4. Testing Memory (Session 1)
**What**: Invoking the chain with a `session_id: "user_1"`.
**Why**: To establish a fact (e.g., "My name is Alice") in the memory of that specific session.
**How**: Passing the `configurable` dictionary in the config. This is where the `session_id` is tracked.

In [31]:
user2 = chain_with_memory.invoke(
    {"input": "What is my name ?"},
    config={"configurable": {"session_id": "user_1"}}
)

c:\Users\santh\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


### 5. Verifying Recall
**What**: Asking a question that requires memory ("What is my name?").
**Why**: To prove that the LLM can retrieve information from a previous turn in the same session.
**How**: By using the same `session_id: "user_1"`, the system automatically fetches "My name is Alice" and feeds it to Gemini, allowing it to answer correctly.